In [ ]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
PAGEINDEX_API_KEY= os.getenv("PAGE_INDEX_KEY")


In [5]:
from pageindex import PageIndexClient
from groq import Groq

pi_client = PageIndexClient(api_key = PAGEINDEX_API_KEY)
groq_client = Groq(api_key= GROQ_API_KEY)


print("Page Index CLient Ready")
print("GROQ Client Ready")

Page Index CLient Ready
GROQ Client Ready


In [8]:
PDF_PATH = "deeplearning.pdf"

print(f"Uploading : {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

Uploading : deeplearning.pdf


In [9]:
print("Building Tree Index")
print("This runs once per document - the index is cached for resuse")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f" Stats: {status}")

    if status == "completed":
        print("\n Tree Index Ready")
        break

    elif status == "failed":
        print("Processing failed...")
        break

    time.sleep(5)

Building Tree Index
This runs once per document - the index is cached for resuse
 Stats: completed

 Tree Index Ready


In [13]:
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f" Top Level sections: {len(pageindex_tree)}")
print("\n Raw tree (first node): ")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))



 Top Level sections: 333

 Raw tree (first node): 
{
  "title": "Fundamentals of Deep Learning",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 1
}


In [ ]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        start, end = node.get("start_index"), node.get("end_index")
        page = "?" if start is None else (start if start == end else f"{start}-{end}")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

In [ ]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

In [21]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "qwen/qwen3.8-27b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM via Groq.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — send only node_id + title (a ToC-style
    # outline). Including each node's text[:150] too (as before) still adds up
    # to 300+ nodes worth of page text, which is what pushed this past Groq's
    # 7000 ITPM limit for this model (the 413 "tokens per minute" error).
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree 
    structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, separators=(",", ":"))}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [22]:
query = "How was data extraction performed"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: How was data extraction performed



BadRequestError: Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': '{\n  "thinking": "The query asks about how data extraction was performed. I need to look for sections related to data processing, extraction, representation, or specific examples where data is pulled from sources (like text for NLP or images for CV). \\n\\n1.  **Chapter 8 (Embedding and Representation Learning)**: This chapter deals with learning representations from data. \'Principal Component Analysis\' (0112), \'Implementing an Autoencoder\' (0114), and \'The Word2Vec Framework\' (0120) involve extracting features/representations from raw data. \\n2.  **Chapter 9 (Models for Sequence Analysis)**: This involves analyzing text. \'Implementing a Part-of-Speech Tagger\' (0128) and \'Dependency Parsing\' (0129) involve extracting linguistic structures

In [ ]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [ ]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        start, end = node.get("start_index"), node.get("end_index")
        page = "?" if start is None else (start if start == end else f"{start}-{end}")
        context_parts.append(
            f"[Section: '{node['title']}' | Page {page}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [ ]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str: 
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [ ]:
answer = vectorless_rag(
    query="What was the conclusion from the review",
    tree=pageindex_tree
)